# 🔬 Notebook 1 — Environment Setup & Dependency Verification
**Pill Counter | Computer Vision Pipeline**

> **Goal:** Verify all dependencies are installed, GPU is accessible, and the project directory structure is correct before any ML work begins.

---
### Notebook Map
| # | Notebook | Description |
|---|----------|-------------|
| **1** | **Environment Setup** | ← You are here |
| 2 | Data Loading & EDA | Dataset discovery, annotation parsing, statistics |
| 3 | Preprocessing & Augmentation | Image transforms, augmentation experiments |
| 4 | Model Training | YOLOv8 training loop, config sweeps |
| 5 | Evaluation & Metrics | mAP, precision-recall, confusion matrix |
| 6 | Real-Time Inference | Camera feed, video inference, FPS benchmarks |


## 1.1 — Install Dependencies
> Re-run this cell only if you're in a fresh environment.

In [ ]:
# %%capture install_output
# Uncomment to install in a fresh environment:
# !pip install ultralytics opencv-python-headless torch torchvision
# !pip install numpy pillow pyyaml scikit-learn matplotlib seaborn
# !pip install ipywidgets tqdm rich

print("Skipping install — assuming environment is already configured.")
print("Uncomment the lines above if you get ImportError below.")


## 1.2 — Import Verification

In [ ]:
import sys
import importlib
from rich.console import Console
from rich.table import Table

console = Console()

deps = [
    ("torch",           "PyTorch"),
    ("cv2",             "OpenCV"),
    ("ultralytics",     "Ultralytics YOLO"),
    ("numpy",           "NumPy"),
    ("PIL",             "Pillow"),
    ("yaml",            "PyYAML"),
    ("sklearn",         "scikit-learn"),
    ("matplotlib",      "Matplotlib"),
    ("seaborn",         "Seaborn"),
]

table = Table(title="Dependency Check")
table.add_column("Package", style="cyan")
table.add_column("Display Name")
table.add_column("Version / Status", style="green")

all_ok = True
for pkg, name in deps:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "installed")
        table.add_row(pkg, name, f"✅  {ver}")
    except ImportError:
        table.add_row(pkg, name, "❌  NOT FOUND", style="red")
        all_ok = False

console.print(table)
if all_ok:
    console.print("\n[bold green]All dependencies satisfied.[/bold green]")
else:
    console.print("\n[bold red]Missing packages — run Section 1.1 to install.[/bold red]")


## 1.3 — GPU / Device Check

In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version    : {torch.version.cuda}")
    print(f"GPU memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = "cuda:0"
else:
    print("⚠️  No GPU detected — inference will be slower on CPU.")
    DEVICE = "cpu"

print(f"\n➡  Active device: {DEVICE}")


## 1.4 — Project Directory Setup
> Creates all required folders if they don't already exist.

In [1]:
from pathlib import Path

DIRS = [
    "data/raw",
    "data/processed/train/images",
    "data/processed/train/labels",
    "data/processed/val/images",
    "data/processed/val/labels",
    "data/processed/test/images",
    "data/processed/test/labels",
    "models/trained",
    "models/exported",
    "results/visualizations",
    "results/metrics",
    "logs",
]

for d in DIRS:
    Path(d).mkdir(parents=True, exist_ok=True)

print("✅  Directory structure ready:")
for d in DIRS:
    print(f"   📁  {d}")


✅  Directory structure ready:
   📁  data/raw
   📁  data/processed/train/images
   📁  data/processed/train/labels
   📁  data/processed/val/images
   📁  data/processed/val/labels
   📁  data/processed/test/images
   📁  data/processed/test/labels
   📁  models/trained
   📁  models/exported
   📁  results/visualizations
   📁  results/metrics
   📁  logs


## 1.5 — Config Loader
> Load `config.yaml` and surface key parameters for downstream notebooks.

In [ ]:
import yaml
from pathlib import Path

CONFIG_PATH = Path("config.yaml")

if not CONFIG_PATH.exists():
    # Write a default config if absent
    default_cfg = {
        "dataset": {"train_size": 0.7, "val_size": 0.15, "test_size": 0.15, "img_size": 640},
        "model":   {"name": "yolov8m", "pretrained": True, "device": 0},
        "training": {"epochs": 20, "batch_size": 8, "learning_rate": 0.001,
                     "patience": 30, "warmup_epochs": 3, "optimizer": "SGD",
                     "weight_decay": 0.0005, "momentum": 0.937},
        "inference": {"confidence_threshold": 0.5, "nms_threshold": 0.45, "max_detections": 100},
        "augmentation": {"hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4,
                         "degrees": 180.0, "translate": 0.15, "scale": 0.5,
                         "flipud": 0.5, "fliplr": 0.5, "mosaic": 1.0},
        "paths": {"data_dir": "data", "raw_data": "data/raw",
                  "processed_data": "data/processed", "trained_models": "models/trained"},
    }
    with open(CONFIG_PATH, "w") as f:
        yaml.dump(default_cfg, f, default_flow_style=False)
    print("⚠️  config.yaml not found — wrote default config.")

with open(CONFIG_PATH) as f:
    CFG = yaml.safe_load(f)

print("✅  Config loaded:")
for section, values in CFG.items():
    print(f"\n  [{section}]")
    if isinstance(values, dict):
        for k, v in values.items():
            print(f"    {k}: {v}")
    else:
        print(f"    {values}")


## ✅  Notebook 1 Complete
> Environment verified. Proceed to **Notebook 2 — Data Loading & EDA**.